# Share-Forge - Train on Colab (T4 GPU)

**One-shot pipeline.** Set `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Walk away for ~15-20 minutes. The final cell triggers a browser download of `checkpoints.zip`.

Pipeline:
1. Clone `github.com/jaivardhandrao/Share-Forge`
2. Apply tasks.py date-range fix (TATAGOLD.NS launched Jan 2024, so old 2018+ ranges were empty)
3. Install runtime deps
4. Fetch + cache TATAGOLD.NS via yfinance (cutoff 2026-03-31)
5. Train LSTM forecaster (PyTorch + Gaussian NLL, ~1 min)
6. Train BC policy (supervised cross-entropy from perfect-hindsight expert, ~30 s)
7. Train PPO RL agent with forecaster signal injected (~10-15 min)
8. Backtest sanity check
9. Zip and download checkpoints

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print(f'torch={torch.__version__}  cuda_available={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

## 2. Clone repo

In [ ]:
%cd /content
!rm -rf Share-Forge
!git clone https://github.com/jaivardhandrao/Share-Forge.git
%cd Share-Forge
import os, sys
sys.path.insert(0, os.getcwd())
print('CWD:', os.getcwd())

## 3. Patch tasks.py with TATAGOLD.NS-compatible date ranges

TATAGOLD.NS only has data from January 2024, so the original 2018+ task ranges return 0 rows. This cell overwrites `server/tasks.py` with realistic ranges. Once everything works in Colab, commit this fix to GitHub on your Mac so future clones are clean.

In [ ]:
%%writefile server/tasks.py
"""Share-Forge Task Definitions (TATAGOLD.NS-compatible date ranges)."""

from __future__ import annotations
from dataclasses import dataclass
from typing import Dict
from models import TaskDifficulty


@dataclass
class TaskSpec:
    task_id: str
    task_type: TaskDifficulty
    start: str
    end: str
    instructions: str
    grading_mode: str
    adversarial_shock: bool = False


EASY_INSTRUCTIONS = 'TATAGOLD.NS post-launch. Long-only daily bars. Action: 0=HOLD, 1=BUY, 2=SELL.'
MEDIUM_VOLATILE_INSTRUCTIONS = 'TATAGOLD.NS volatile regime. Optimize Sharpe.'
MEDIUM_SIDEWAYS_INSTRUCTIONS = 'TATAGOLD.NS sideways. Sharpe minus turnover.'
HARD_INSTRUCTIONS = 'TATAGOLD.NS with synthetic shocks. 0.4*Sharpe + 0.3*Calmar + 0.3*(1-MaxDD).'


TASKS: Dict[TaskDifficulty, TaskSpec] = {
    TaskDifficulty.EASY_LONG_ONLY: TaskSpec(
        task_id='a1b2c3d4-1111-4000-a000-000000000001',
        task_type=TaskDifficulty.EASY_LONG_ONLY,
        start='2024-02-15', end='2024-08-31',
        instructions=EASY_INSTRUCTIONS, grading_mode='total_return',
    ),
    TaskDifficulty.MEDIUM_VOLATILE: TaskSpec(
        task_id='a1b2c3d4-2222-4000-a000-000000000002',
        task_type=TaskDifficulty.MEDIUM_VOLATILE,
        start='2024-09-01', end='2025-02-28',
        instructions=MEDIUM_VOLATILE_INSTRUCTIONS, grading_mode='sharpe',
    ),
    TaskDifficulty.MEDIUM_SIDEWAYS: TaskSpec(
        task_id='a1b2c3d4-3333-4000-a000-000000000003',
        task_type=TaskDifficulty.MEDIUM_SIDEWAYS,
        start='2025-03-01', end='2025-08-31',
        instructions=MEDIUM_SIDEWAYS_INSTRUCTIONS, grading_mode='sharpe_turnover',
    ),
    TaskDifficulty.HARD_ADVERSARIAL: TaskSpec(
        task_id='a1b2c3d4-4444-4000-a000-000000000004',
        task_type=TaskDifficulty.HARD_ADVERSARIAL,
        start='2025-09-01', end='2026-03-31',
        instructions=HARD_INSTRUCTIONS, grading_mode='composite',
        adversarial_shock=True,
    ),
}


def get_task(task_type: TaskDifficulty) -> TaskSpec:
    return TASKS[task_type]


## 4. Install dependencies

Colab already has PyTorch with CUDA — install only what's missing.

In [ ]:
# Install non-torch deps separately, then SB3 with --no-deps so it can't replace
# Colab's preinstalled CUDA-enabled torch with a CPU-only wheel.
!pip install -q yfinance gymnasium fastapi uvicorn pydantic gradio plotly tensorboard \
    sqlalchemy 'openenv-core>=0.2.0' cloudpickle pandas matplotlib
!pip install -q --no-deps 'stable-baselines3>=2.3.0' 'sb3-contrib>=2.3.0'

import torch
print(f'torch={torch.__version__}  cuda_available={torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'CUDA torch was overwritten — run: !pip install --upgrade --force-reinstall --no-deps torch --index-url https://download.pytorch.org/whl/cu121'

## 5. Fetch + cache TATAGOLD.NS

In [ ]:
!python -m server.data_loader
from server.data_loader import load, load_live
df = load(); live = load_live()
print(f'Train rows: {len(df)}  range: {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'Live  rows: {len(live)}  range: {live["date"].min().date() if not live.empty else "-"} -> {live["date"].max().date() if not live.empty else "-"}')

## 6. (Optional) Live TensorBoard

Safe to run inline. If you skipped this, you can launch later with `%tensorboard --logdir runs`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 7. Stage 1 - Train LSTM forecaster (~1 min on T4)

In [ ]:
!python train_forecaster.py \
    --epochs 50 --batch-size 64 \
    --horizon 5 --window-size 20 \
    --hidden-size 64 --num-layers 2 \
    --device cuda

## 8. Stage 2 - Train BC policy (~30 s on T4)

In [ ]:
!python train_bc.py \
    --epochs 30 --batch-size 64 \
    --horizon 5 \
    --buy-threshold 0.005 --sell-threshold -0.005 \
    --device cuda

## 9. Stage 3 - PPO RL fine-tune with forecaster signal (~10-15 min on T4)

In [ ]:
!python train.py \
    --task easy_long_only \
    --timesteps 200000 \
    --device cuda \
    --use-ml-forecaster

## 10. Sanity-check backtest

Replays `easy_long_only` with the trained PPO agent and plots equity vs. buy-and-hold.

In [ ]:
import matplotlib.pyplot as plt
from server.data_loader import load, slice_by_dates
from server.trading_env import ShareForgeTradingEnv, TradingConfig
from server.policy_loader import predict, policy_status
from server.tasks import get_task
from models import TaskDifficulty

print('Policy status:', policy_status())

spec = get_task(TaskDifficulty.EASY_LONG_ONLY)
df = slice_by_dates(load(), spec.start, spec.end).reset_index(drop=True)
env = ShareForgeTradingEnv(df, TradingConfig(use_ml_forecaster=True))
obs, _ = env.reset()
n_extra = 3 if env._forecast_series is not None else 2

done = False
while not done:
    raw_window = obs[:, :-n_extra].tolist()
    action, _ = predict(raw_window, is_long=env._is_long)
    obs, _, term, trunc, _ = env.step(int(action))
    done = term or trunc

equity = env.equity_curve
prices = df['close'].to_numpy()
bh = (env.config.initial_cash * (prices / prices[0]))[: len(equity)]

plt.figure(figsize=(12, 5))
plt.plot(equity, label='Agent')
plt.plot(bh, label='Buy & Hold', linestyle='--')
plt.legend(); plt.grid(alpha=0.3); plt.title(f'Backtest: {spec.task_type.value} ({spec.start} -> {spec.end})')
plt.xlabel('Step'); plt.ylabel('Portfolio Value (INR)')
plt.show()
print(f'Final equity: {equity[-1]:,.0f}  (B&H: {bh[-1]:,.0f})  trades: {env._n_trades}')

## 11. Bundle + download checkpoints

Triggers a browser download of `checkpoints.zip`. Drop it into your local repo's `checkpoints/` folder.

In [ ]:
!ls -la checkpoints/
!zip -r checkpoints.zip checkpoints/
from google.colab import files
files.download('checkpoints.zip')